In [2]:
import requests
import json
import numpy as np
import pandas as pd
import pytz
from datetime import datetime, timedelta
import concurrent.futures
import base64
import os
import time

import duckdb

DB_PATH = r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\duckdb\nhl_data.duckdb"

con = duckdb.connect(DB_PATH)
print("Connected:", DB_PATH)

# try:
#     BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
# except NameError:
#     BASE_DIR = os.getcwd()

# DB_PATH = os.path.join(BASE_DIR, "duckdb", "nhl_data.duckdb")
# con = duckdb.connect(DB_PATH)

import random
# Suppress SettingWithCopyWarning
pd.options.mode.chained_assignment = None
pd.set_option("display.max_columns", None)

Connected: C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\duckdb\nhl_data.duckdb


## Completed Seasons Data

In [3]:

# Season end dates (same format as your example)
season_end_dates = {
    # "20132014": "2015-04-13",
    # "20142015": "2015-04-11",
    # "20152016": "2016-04-10",
    # "20162017": "2017-04-09",
    "20172018": "2018-04-08",
    "20182019": "2019-04-06",
    "20192020": "2020-03-12",
    "20202021": "2021-05-19",
    "20212022": "2022-04-29",
    "20222023": "2023-04-13",
    "20232024": "2024-04-18",
    "20242025": "2025-04-17"
}

base_url = "https://api.nhle.com/stats/rest/en/team/penaltykill"
all_seasons_data = []

for season_id in season_end_dates.keys():
    print(f"Fetching PK data for {season_id}...")

    start = 0
    limit = 100
    all_results = []

    while True:

        params = {
            "isAggregate": "false",
            "isGame": "false",
            "sort": '[{"property":"penaltyKillPct","direction":"DESC"},{"property":"teamId","direction":"ASC"}]',
            "cayenneExp": f"gamesPlayed>=1 and gameTypeId=2 and seasonId={season_id}",
            "start": start,
            "limit": limit
        }

        response = requests.get(base_url, params=params)

        # Error check
        if response.status_code != 200:
            print(f"Error fetching season {season_id}: {response.status_code}")
            break

        data = response.json().get("data", [])

        if not data:
            break  # End pagination

        all_results.extend(data)
        start += limit

    # Build DF per season
    if all_results:
        df = pd.DataFrame(all_results)
        df["season"] = season_id
        all_seasons_data.append(df)

# Combine into one DataFrame
if all_seasons_data:
    pk_data_df = pd.concat(all_seasons_data, ignore_index=True)
else:
    pk_data_df = pd.DataFrame()




export_path = r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Team Stats\pk_agg_data.csv"

# Ensure the folder exists before writing the file
os.makedirs(os.path.dirname(export_path), exist_ok=True)

pk_data_df.to_csv(export_path, index=False)

pk_data_df.head()

Fetching PK data for 20172018...
Fetching PK data for 20182019...
Fetching PK data for 20192020...
Fetching PK data for 20202021...
Fetching PK data for 20212022...
Fetching PK data for 20222023...
Fetching PK data for 20232024...
Fetching PK data for 20242025...


,gamesPlayed,losses,otLosses,penaltyKillNetPct,penaltyKillPct,pkNetGoals,pkNetGoalsPerGame,pkTimeOnIcePerGame,points,pointsPct,ppGoalsAgainst,ppGoalsAgainstPerGame,seasonId,shGoalsFor,shGoalsForPerGame,teamFullName,teamId,ties,timesShorthanded,timesShorthandedPerGame,wins,season
0,82,29,8,0.869230,0.85000,-34,-0.41463,324.195121,98,0.0,39,0.0,20172018,5,0.060975,Los Angeles Kings,26,None,260,3.17073,45,20172018
1,82,27,10,0.883928,0.84821,-26,-0.31707,265.817073,100,0.0,34,0.0,20172018,8,0.097560,San Jose Sharks,28,None,224,2.73170,45,20172018
2,82,20,12,0.873469,0.83673,-31,-0.37804,303.707317,112,0.0,40,0.0,20172018,9,0.109756,Boston Bruins,6,None,245,2.98780,50,20172018
3,82,30,9,0.858736,0.83271,-38,-0.46341,332.634146,95,0.0,45,0.0,20172018,7,0.085365,Colorado Avalanche,21,None,269,3.28048,43,20172018
4,82,25,13,0.868613,0.83211,-36,-0.43902,343.097560,101,0.0,46,0.0,20172018,10,0.121951,Anaheim Ducks,24,None,274,3.34146,44,20172018


In [3]:


# Your season list
season_end_dates = {
    # "20132014": "2015-04-13",
    # "20142015": "2015-04-11",
    # "20152016": "2016-04-10",
    # "20162017": "2017-04-09",
    "20172018": "2018-04-08",
    "20182019": "2019-04-06",
    "20192020": "2020-03-12",
    "20202021": "2021-05-19",
    "20212022": "2022-04-29",
    "20222023": "2023-04-13",
    "20232024": "2024-04-18",
    "20242025": "2025-04-17"
}

base_url = "https://api.nhle.com/stats/rest/en/team/powerplay"
all_seasons_data = []

for season_id in season_end_dates.keys():
    print(f"Fetching PP data for {season_id}...")

    start = 0
    limit = 100
    all_results = []

    while True:
        params = {
            "isAggregate": "false",
            "isGame": "false",
            "sort": '[{"property":"powerPlayPct","direction":"DESC"},{"property":"teamId","direction":"ASC"}]',
            "cayenneExp": f"gamesPlayed>=1 and gameTypeId=2 and seasonId={season_id}",
            "start": start,
            "limit": limit
        }

        response = requests.get(base_url, params=params)

        # Error handling
        if response.status_code != 200:
            print(f"Error fetching PP data for {season_id}: {response.status_code}")
            break

        data = response.json().get("data", [])
        if not data:
            break  # Pagination ended

        all_results.extend(data)
        start += limit

    # Add season tag + append
    if all_results:
        df = pd.DataFrame(all_results)
        df["season"] = season_id
        all_seasons_data.append(df)

# Final combined DF
if all_seasons_data:
    pp_data_df = pd.concat(all_seasons_data, ignore_index=True)
else:
    pp_data_df = pd.DataFrame()



export_path = r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Team Stats\pp_agg_data.csv"

# Ensure the folder exists before writing the file
os.makedirs(os.path.dirname(export_path), exist_ok=True)

pp_data_df.to_csv(export_path, index=False)

pp_data_df.head()


Fetching PP data for 20172018...
Fetching PP data for 20182019...
Fetching PP data for 20192020...
Fetching PP data for 20202021...
Fetching PP data for 20212022...
Fetching PP data for 20222023...
Fetching PP data for 20232024...
Fetching PP data for 20242025...


,gamesPlayed,losses,otLosses,pointPct,points,powerPlayGoalsFor,powerPlayNetPct,powerPlayPct,ppGoalsPerGame,ppNetGoals,ppNetGoalsPerGame,ppOpportunities,ppOpportunitiesPerGame,ppTimeOnIcePerGame,seasonId,shGoalsAgainst,shGoalsAgainstPerGame,teamFullName,teamId,ties,wins,season
0,82,29,6,0.0,100,68,0.250000,0.26153,0.0,65,0.79268,260,3.17073,304.158536,20172018,3,0.036585,Pittsburgh Penguins,5,None,47,20172018
1,82,26,7,0.0,105,56,0.227678,0.25000,0.0,51,0.62195,224,2.73170,258.560975,20172018,5,0.060975,Toronto Maple Leafs,10,None,49,20172018
2,82,23,5,0.0,113,66,0.228260,0.23913,0.0,63,0.76829,276,3.36585,326.341463,20172018,3,0.036585,Tampa Bay Lightning,14,None,54,20172018
3,82,20,12,0.0,112,61,0.196911,0.23552,0.0,51,0.62195,259,3.15853,305.268292,20172018,10,0.121951,Boston Bruins,6,None,50,20172018
4,82,20,10,0.0,114,64,0.208029,0.23357,0.0,57,0.69512,274,3.34146,316.695121,20172018,7,0.085365,Winnipeg Jets,52,None,52,20172018


## Game by Game - includes current season

In [3]:
import requests
import time
import random
import os
import pandas as pd

# =========================================
# SEASON DATE RANGES
# =========================================
season_start_dates = {
    # "20132014": "2013-10-01",
    # "20142015": "2014-10-08",
    # "20152016": "2015-10-07",
    # "20162017": "2016-10-12",
    # "20172018": "2017-10-04",
    # "20182019": "2018-10-04",
    # "20192020" :  "2019-10-02",
    # "20202021" : "2021-01-13",
    # "20212022": "2021-10-12",
    # "20222023": "2022-10-11",
    # "20232024": "2023-10-11",
    # "20242025": "2024-10-08",
    "20252026": "2025-11-27"
}

season_end_dates = {
    # "20132014": "2014-04-13",
    # "20142015": "2015-04-11",
    # "20152016": "2016-04-10",
    # "20162017": "2017-04-09",
    # "20172018": "2018-04-08",
    # "20182019": "2019-04-06",
    # "20192020": "2020-03-12",
    # "20202021": "2021-05-19"
    # "20212022": "2022-04-29",
    # "20222023": "2023-04-13",
    # "20232024": "2024-04-18",
    # "20242025": "2025-04-17",
    "20252026": "2025-12-05"
}

# =========================================
# SAFE REQUEST (429 BACKOFF)
# =========================================
def safe_request(url, params, max_retries=5):
    backoff = 1.0
    for attempt in range(max_retries):
        resp = requests.get(url, params=params)

        if resp.status_code == 200:
            return resp

        if resp.status_code == 429:  # rate limited
            print(f"⚠️ 429 Rate limit. Waiting {backoff:.1f}s...")
            time.sleep(backoff)
            backoff *= 2
            continue

        print(f"⚠️ Error {resp.status_code}. Retrying in {backoff:.1f}s...")
        time.sleep(backoff)
        backoff *= 2

    raise Exception(f"Failed after {max_retries} retries: {resp.status_code}")

# =========================================
# PK SCRAPER — CORRECT PAGINATION (start+=100)
# =========================================
def get_pk_by_game(start_date, end_date, base_delay=1.0):

    url = "https://api.nhle.com/stats/rest/en/team/penaltykill"
    limit = 100  # NHL API always returns 100 rows per page

    cayenne = (
        f'gameDate<="{end_date} 23:59:59" '
        f'and gameDate>="{start_date}" '
        f'and gameTypeId=2'
    )

    params = {
        "isAggregate": "false",
        "isGame": "true",
        "sort": '[{"property":"teamFullName","direction":"ASC"},{"property":"teamId","direction":"ASC"}]',
        "start": 0,
        "limit": limit,
        "cayenneExp": cayenne,
    }

    # -----------------------------------------
    # First call → get total row count
    # -----------------------------------------
    resp = safe_request(url, params)
    payload = resp.json()

    if "total" not in payload:
        raise ValueError("API response missing 'total' — cannot paginate.")

    total = payload["total"]
    print(f"  → API reports total rows = {total}")

    all_rows = []
    start = 0

    # -----------------------------------------
    # Loop through all pages
    # -----------------------------------------
    while start < total:

        params["start"] = start
        resp = safe_request(url, params)
        page = resp.json().get("data", [])

        print(f"     Fetching rows {start} to {start+limit} → got {len(page)} rows")

        if not page:
            print("⚠️ Empty page before reaching total — stopping early.")
            break

        all_rows.extend(page)

        start += limit   # increment by 100

        time.sleep(base_delay + random.uniform(0.3, 0.6))

    # Just in case API duplicates rows
    if len(all_rows) > total:
        all_rows = all_rows[:total]

    print(f"  → Downloaded {len(all_rows)} rows (expected {total})")

    return pd.DataFrame(all_rows)

# =========================================
# MULTI-SEASON LOOP
# =========================================
pk_all = []

for season_id, start_date in season_start_dates.items():
    end_date = season_end_dates[season_id]

    print(f"\n=== PK {season_id}: {start_date} → {end_date} ===")

    df = get_pk_by_game(start_date, end_date)

    if not df.empty:
        df["season"] = season_id
        pk_all.append(df)

        team_count = df["teamId"].nunique()
        print(f"  → Retrieved {team_count} unique teams")

        if season_id < "20212022" and team_count < 31:
            print("  ⚠️ Missing teams in a 31-team season!")
        if season_id >= "20212022" and team_count < 32:
            print("  ⚠️ Missing teams in a 32-team season!")





=== PK 20252026: 2025-11-27 → 2025-12-05 ===
  → API reports total rows = 120
     Fetching rows 0 to 100 → got 100 rows
     Fetching rows 100 to 200 → got 20 rows
  → Downloaded 120 rows (expected 120)
  → Retrieved 32 unique teams


In [22]:

# =========================================
# FINAL OUTPUT
# =========================================
pk_by_game_all = pd.concat(pk_all, ignore_index=True)
pk_by_game_all=pk_by_game_all.rename(columns={'season':'season_id', 'gameId':'game_id','teamId':'team_id'})
# Convert to datetime → .dt.date so DuckDB loads it as DATE
pk_by_game_all["gameDate"] = (
    pd.to_datetime(pk_by_game_all["gameDate"], errors="coerce").dt.date
)

pk_by_game_all["season_id"] = pk_by_game_all["season_id"].astype("Int64")
pk_by_game_all["game_id"] = pk_by_game_all["game_id"].astype("Int64")
pk_by_game_all["team_id"] = pk_by_game_all["team_id"].astype("Int64")
pk_by_game_all.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 25 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   gameDate                 120 non-null    object 
 1   game_id                  120 non-null    Int64  
 2   gamesPlayed              120 non-null    int64  
 3   homeRoad                 120 non-null    object 
 4   losses                   120 non-null    int64  
 5   opponentTeamAbbrev       120 non-null    object 
 6   otLosses                 120 non-null    int64  
 7   penaltyKillNetPct        119 non-null    float64
 8   penaltyKillPct           119 non-null    float64
 9   pkNetGoals               120 non-null    int64  
 10  pkNetGoalsPerGame        120 non-null    float64
 11  pkTimeOnIcePerGame       120 non-null    float64
 12  points                   120 non-null    int64  
 13  pointsPct                120 non-null    float64
 14  ppGoalsAgainst           1

In [ ]:
correct_order = [
    "season_id",
    "game_id",
    "team_id",
    "gameDate",
    "gamesPlayed",
    "homeRoad",
    "losses",
    "opponentTeamAbbrev",
    "otLosses",
    "penaltyKillNetPct",
    "penaltyKillPct",
    "pkNetGoals",
    "pkNetGoalsPerGame",
    "pkTimeOnIcePerGame",
    "points",
    "pointsPct",
    "ppGoalsAgainst",
    "ppGoalsAgainstPerGame",
    "shGoalsFor",
    "shGoalsForPerGame",
    "teamFullName",
    "ties",
    "timesShorthanded",
    "timesShorthandedPerGame",
    "wins"
]

# Reorder columns
pk_by_game_all = pk_by_game_all[correct_order]
pk_by_game_all["gameDate"] = pd.to_datetime(pk_by_game_all["gameDate"]).dt.date

# Insert new records
con.execute("""
    INSERT INTO all_pk
    SELECT * FROM pk_by_game_all
""")



In [ ]:


# pk_by_game_all = pd.concat([pk_by_game_all,update_pk])


# pk_by_game_all = pk_by_game_all.sort_values(by=['gameId'])

# export_path = r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Team Stats\pk_by_game_all.csv"
# os.makedirs(os.path.dirname(export_path), exist_ok=True)

# pk_by_game_all.to_csv(export_path, index=False)

# print("\n✅ Export complete!")


,column_name,column_type,null,key,default,extra
0,season_id,BIGINT,YES,None,None,None
1,game_id,BIGINT,YES,None,None,None
2,team_id,BIGINT,YES,None,None,None
3,gameDate,DATE,YES,None,None,None
4,gamesPlayed,BIGINT,YES,None,None,None
5,homeRoad,VARCHAR,YES,None,None,None
6,losses,BIGINT,YES,None,None,None
7,opponentTeamAbbrev,VARCHAR,YES,None,None,None
8,otLosses,BIGINT,YES,None,None,None
9,penaltyKillNetPct,DOUBLE,YES,None,None,None


In [25]:
import requests
import time
import random
import os
import pandas as pd

# =========================================
# SEASON DATE RANGES
# =========================================
season_start_dates = {
    # "20132014": "2013-10-01",
    # "20142015": "2014-10-08",
    # "20152016": "2015-10-07",
    # "20162017": "2016-10-12",
    # "20172018": "2017-10-04",
    # "20182019": "2018-10-04",
    # "20192020" :  "2019-10-02",
    # "20202021" : "2021-01-13",
    # "20212022": "2021-10-12",
    # "20222023": "2022-10-11",
    # "20232024": "2023-10-11",
    # "20242025": "2024-10-08",
    "20252026": "2025-11-27"
}

season_end_dates = {
    # "20132014": "2014-04-13",
    # "20142015": "2015-04-11",
    # "20152016": "2016-04-10",
    # "20162017": "2017-04-09",
    # "20172018": "2018-04-08",
    # "20182019": "2019-04-06",
    # "20192020": "2020-03-12",
    # "20202021": "2021-05-19"
    # "20212022": "2022-04-29",
    # "20222023": "2023-04-13",
    # "20232024": "2024-04-18",
    # "20242025": "2025-04-17",
    "20252026": "2025-12-05"
}

# =========================================
# SAFE REQUEST (429 BACKOFF)
# =========================================
def safe_request(url, params, max_retries=5):
    backoff = 1.0
    for attempt in range(max_retries):
        resp = requests.get(url, params=params)

        if resp.status_code == 200:
            return resp

        if resp.status_code == 429:
            print(f"⚠️ 429 Rate limit. Waiting {backoff:.1f}s...")
            time.sleep(backoff)
            backoff *= 2
            continue

        print(f"⚠️ Error {resp.status_code}. Retrying in {backoff:.1f}s...")
        time.sleep(backoff)
        backoff *= 2

    raise Exception(f"Failed after {max_retries} retries: {resp.status_code}")

# =========================================
# POWER PLAY SCRAPER — FULL PAGINATION
# =========================================
def get_pp_by_game(start_date, end_date, base_delay=1.0):
    url = "https://api.nhle.com/stats/rest/en/team/powerplay"
    limit = 100  # NHL API returns 100 rows per page

    cayenne = (
        f'gameDate<="{end_date} 23:59:59" '
        f'and gameDate>="{start_date}" '
        f'and gameTypeId=2'
    )

    params = {
        "isAggregate": "false",
        "isGame": "true",
        "sort": '[{"property":"powerPlayPct","direction":"DESC"},{"property":"teamId","direction":"ASC"}]',
        "start": 0,
        "limit": limit,
        "cayenneExp": cayenne
    }

    # -------- First call to get total rows --------
    resp = safe_request(url, params)
    payload = resp.json()

    if "total" not in payload:
        raise Exception("API missing 'total' field — cannot paginate")

    total = payload["total"]
    print(f"  → API reports total rows = {total}")

    all_rows = []
    start = 0

    # -------- Loop pages until all rows fetched --------
    while start < total:
        params["start"] = start
        resp = safe_request(url, params)
        page = resp.json().get("data", [])

        print(f"     Fetching rows {start} to {start+limit} → got {len(page)} rows")

        if not page:
            print("⚠️ Empty page before reaching total — stopping early.")
            break

        all_rows.extend(page)
        start += limit

        time.sleep(base_delay + random.uniform(0.25, 0.5))

    # Trim duplicates if any extra
    if len(all_rows) > total:
        all_rows = all_rows[:total]

    print(f"  → Downloaded {len(all_rows)} rows (expected {total})")
    return pd.DataFrame(all_rows)

# =========================================
# MULTI-SEASON LOOP
# =========================================
pp_all = []

for season_id, start_date in season_start_dates.items():
    end_date = season_end_dates[season_id]

    print(f"\n=== PP {season_id}: {start_date} → {end_date} ===")
    df = get_pp_by_game(start_date, end_date)

    if not df.empty:
        df["season"] = season_id
        pp_all.append(df)



=== PP 20252026: 2025-11-27 → 2025-12-05 ===
  → API reports total rows = 120
     Fetching rows 0 to 100 → got 100 rows
     Fetching rows 100 to 200 → got 20 rows
  → Downloaded 120 rows (expected 120)


In [29]:
# update_this = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Team Stats\pp_by_game_all.csv")

# =========================================
# FINAL OUTPUT
# =========================================
pp_all_game = pd.concat(pp_all, ignore_index=True)
pp_all_game.head()



,gameDate,gameId,gamesPlayed,homeRoad,losses,opponentTeamAbbrev,otLosses,pointPct,points,powerPlayGoalsFor,powerPlayNetPct,powerPlayPct,ppGoalsPerGame,ppNetGoals,ppNetGoalsPerGame,ppOpportunities,ppOpportunitiesPerGame,ppTimeOnIcePerGame,shGoalsAgainst,shGoalsAgainstPerGame,teamFullName,teamId,ties,wins,season
0,2025-12-03,2025020422,1,H,0,WPG,0,1.0,2,1,1.00,1.00,1.0,1,1.0,1,1.0,28.0,0,0.0,Montréal Canadiens,8,None,1,20252026
1,2025-11-28,2025020377,1,R,0,DET,0,1.0,2,1,1.00,1.00,1.0,1,1.0,1,1.0,36.0,0,0.0,Tampa Bay Lightning,14,None,1,20252026
2,2025-11-29,2025020393,1,R,0,SEA,0,1.0,2,2,1.00,1.00,2.0,2,2.0,2,2.0,102.0,0,0.0,Edmonton Oilers,22,None,1,20252026
3,2025-12-04,2025020434,1,H,0,SEA,0,1.0,2,4,0.80,0.80,4.0,4,4.0,5,5.0,217.0,0,0.0,Edmonton Oilers,22,None,1,20252026
4,2025-12-01,2025020407,1,R,0,PHI,0,1.0,2,3,0.75,0.75,3.0,3,3.0,4,4.0,237.0,0,0.0,Pittsburgh Penguins,5,None,1,20252026


In [30]:

pp_all_game=pp_all_game.rename(columns={'season':'season_id', 'gameId':'game_id','teamId':'team_id'})
# Convert to datetime → .dt.date so DuckDB loads it as DATE
pp_all_game["gameDate"] = (
    pd.to_datetime(pp_all_game["gameDate"], errors="coerce").dt.date
)

pp_all_game["season_id"] = pp_all_game["season_id"].astype("Int64")
pp_all_game["game_id"] = pp_all_game["game_id"].astype("Int64")
pp_all_game["team_id"] = pp_all_game["team_id"].astype("Int64")
pp_all_game.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 25 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   gameDate                120 non-null    object 
 1   game_id                 120 non-null    Int64  
 2   gamesPlayed             120 non-null    int64  
 3   homeRoad                120 non-null    object 
 4   losses                  120 non-null    int64  
 5   opponentTeamAbbrev      120 non-null    object 
 6   otLosses                120 non-null    int64  
 7   pointPct                120 non-null    float64
 8   points                  120 non-null    int64  
 9   powerPlayGoalsFor       120 non-null    int64  
 10  powerPlayNetPct         119 non-null    float64
 11  powerPlayPct            119 non-null    float64
 12  ppGoalsPerGame          120 non-null    float64
 13  ppNetGoals              120 non-null    int64  
 14  ppNetGoalsPerGame       120 non-null    fl

In [32]:
con.execute("DESCRIBE all_pp").fetchdf()


,column_name,column_type,null,key,default,extra
0,season_id,BIGINT,YES,None,None,None
1,game_id,BIGINT,YES,None,None,None
2,team_id,BIGINT,YES,None,None,None
3,gameDate,DATE,YES,None,None,None
4,gamesPlayed,BIGINT,YES,None,None,None
5,homeRoad,VARCHAR,YES,None,None,None
6,losses,BIGINT,YES,None,None,None
7,opponentTeamAbbrev,VARCHAR,YES,None,None,None
8,otLosses,BIGINT,YES,None,None,None
9,pointPct,DOUBLE,YES,None,None,None


In [33]:
correct_order = [
    "season_id",
    "game_id",
    "team_id",
    "gameDate",
    "gamesPlayed",
    "homeRoad",
    "losses",
    "opponentTeamAbbrev",
    "otLosses",
    "pointPct",
    "points",
    "powerPlayGoalsFor",
    "powerPlayNetPct",
    "powerPlayPct",
    "ppGoalsPerGame",
    "ppNetGoals",
    "ppNetGoalsPerGame",
    "ppOpportunities",
    "ppOpportunitiesPerGame",
    "ppTimeOnIcePerGame",
    "shGoalsAgainst",
    "shGoalsAgainstPerGame",
    "teamFullName",
    "ties",
    "wins"
]

# Reorder columns
pp_all_game = pp_all_game[correct_order]
pp_all_game["gameDate"] = pd.to_datetime(pp_all_game["gameDate"]).dt.date

# Insert new records
con.execute("""
    INSERT INTO all_pp
    SELECT * FROM pp_all_game
""")



In [35]:
con.execute("SELECT * FROM all_pp").fetchdf().tail()

con.close()
print("DuckDB connection closed.")

DuckDB connection closed.


In [ ]:

# # =========================================
# # FINAL OUTPUT
# # =========================================
# pp_by_game_all = pd.concat(pp_all, ignore_index=True)
# pp_by_game_all = pd.concat([pp_by_game_all,update_this ])

# pp_by_game_all = pp_by_game_all.sort_values(by=['gameId'])

# export_path = r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Team Stats\pp_by_game_all.csv"
# os.makedirs(os.path.dirname(export_path), exist_ok=True)

# pp_by_game_all.to_csv(export_path, index=False)

# print("\n✅ PP export complete!")



✅ PP export complete!
